# Eksperimen Tahap 1 & 2: Preprocessing, Chunking, & Ekstraksi Vektor

Notebook ini berfokus pada alur terintegrasi:
1. Baca PDF secara langsung dari URL Public (Supabase Storage).
2. Ekstrak teks secara cerdas (pisahkan bagian Atas/Bawah).
3. Panggil Mistral LLM dengan prompt ketat (Strict Prompt) agar tidak bingung membedakan Header & Footer.
4. Bersihkan teks berbasis **Regex Lanjutan** dan pendeteksian pola LLM.
5. Uji Pemotongan Teks (Chunking) untuk: `Character`, `Word (Token)`, dan `Sentence`.
6. Embedding menggunakan `BAAI/bge-small-en-v1.5`.
7. Menyimpan / Cek Eksistensi Jurnal beserta Vektor Chunk nya secara utuh di Supabase pgvector.

In [ ]:
!pip install PyMuPDF langchain-text-splitters supabase nltk requests python-dotenv sentence-transformers pandas

In [ ]:
import os
import re
import json
import io
import requests
import time
import fitz  # PyMuPDF
import nltk
import pandas as pd
from supabase import create_client, Client
from langchain_text_splitters import TokenTextSplitter, RecursiveCharacterTextSplitter, CharacterTextSplitter
from sentence_transformers import SentenceTransformer

nltk.download('punkt')

# --- CONFIGURATION ---
SUPABASE_URL = "https://YOUR_PROJECT_REF.supabase.co"
SUPABASE_KEY = "YOUR_SUPABASE_SERVICE_ROLE_KEY"
MISTRAL_API_KEY = "YOUR_MISTRAL_API_KEY"

# Gunakan Client Supabase dengan Service Role Key untuk operasi Insert/DB
try:
    supabase_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
except Exception as e:
    supabase_client = None
    print("Supabase Client belum dikonfigurasi / Error:", e)

### 1. Ekstraksi Pola (Header/Footer) & Teks dari URL PDF

In [ ]:
def download_and_extract_text(pdf_url):
    print(f"Mengunduh PDF dari: {pdf_url} ...")
    response = requests.get(pdf_url)
    response.raise_for_status()
    
    pdf_stream = io.BytesIO(response.content)
    doc = fitz.open(stream=pdf_stream, filetype="pdf")
    
    pages_text = []
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        # Get text secara biasa
        pages_text.append(page.get_text("text"))
        
    return pages_text

def get_mistral_analysis(system_prompt, user_content):
    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {MISTRAL_API_KEY}"
    }
    data = {
        "model": "mistral-small-latest",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ],
        "temperature": 0.1,
        "max_tokens": 300
    }
    
    response = requests.post(url, headers=headers, json=data)
    response.raise_for_status()
    res_json = response.json()
    content = res_json['choices'][0]['message']['content'].strip()
    
    content = re.sub(r'```json', '', content)
    content = re.sub(r'```', '', content)
    return json.loads(content)

def extract_metadata_and_patterns(pages_text):
    metadata = {"title": "", "authors": "", "year": "", "header_even": "", "footer_even": "", "header_odd": "", "footer_odd": ""}
    sys_prompt = "You are an API. Return ONLY a valid JSON object with no markdown."
    
    # 1. Hal 1: Metadata
    if len(pages_text) > 0:
        print("\n--- Ekstraksi Metadata (Halaman 1) ---")
        user_prompt = f"""Extract metadata from this academic paper's first page:
{pages_text[0][:2500]}

Extract:
1. title: Full paper title
2. authors: Author names
3. year: Publication year (4 digits)

Return format: {{\"title\":\"...\",\"authors\":\"...\",\"year\":\"...\"}}"""
        try:
            metadata.update(get_mistral_analysis(sys_prompt, user_prompt))
        except Exception as e: print("Gagal metadata:", e)
            
    def get_top_bottom_snippets(page_text):
        if len(page_text) < 1000: return page_text
        # Memberikan margin yang sangat ekstrim agar Mistral tidak tersesat di teks tengah
        return f"[TOP OF PAGE]\n{page_text[:300]}\n...\n[BOTTOM OF PAGE]\n{page_text[-400:]}"

    even_pages = []
    if len(pages_text) > 1: even_pages.append(get_top_bottom_snippets(pages_text[1]))
    if len(pages_text) > 3: even_pages.append(get_top_bottom_snippets(pages_text[3]))
    
    header_footer_prompt = """Analyze this text from an academic paper. Identify the REPEATING HEADER and FOOTER elements.
RULES:
1. 'headerPattern': Exact text found ONLY in the [TOP OF PAGE] section. Read from the top line. Max 15 words. If none, return empty string.
2. 'footerPattern': Exact text found ONLY in the [BOTTOM OF PAGE] section. Usually things like Journal Name, Volume, Year, Download links, DOI, Licenses. It might be very long. Do not summarize, exact text only.
IMPORTANT EXCLUSIONS:
- Do NOT extract paragraph text, statistical notes, or body copies.
- Do NOT extract table/figure captions.
- Do NOT confuse headers and footers.

Text Snippets:
{text_snippets}

Return format: {{\"headerPattern\":\"...\", \"footerPattern\":\"...\"}}"""

    if even_pages:
        print("\n--- Mencari Pola Halaman Genap ---")
        try:
            res_even = get_mistral_analysis(sys_prompt, header_footer_prompt.format(text_snippets="\n\n===NEXT PAGE===\n\n".join(even_pages)))
            metadata['header_even'] = res_even.get('headerPattern', '')
            metadata['footer_even'] = res_even.get('footerPattern', '')
        except Exception as e: print("Gagal ola genap:", e)

    odd_pages = []
    if len(pages_text) > 2: odd_pages.append(get_top_bottom_snippets(pages_text[2]))
    if len(pages_text) > 4: odd_pages.append(get_top_bottom_snippets(pages_text[4]))
    
    if odd_pages:
        print("\n--- Mencari Pola Halaman Ganjil ---")
        try:
            res_odd = get_mistral_analysis(sys_prompt, header_footer_prompt.format(text_snippets="\n\n===NEXT PAGE===\n\n".join(odd_pages)))
            metadata['header_odd'] = res_odd.get('headerPattern', '')
            metadata['footer_odd'] = res_odd.get('footerPattern', '')
        except Exception as e: print("Gagal pola ganjil:", e)

    return metadata


### 1.5. Validasi Pola (MANUAL INPUT)

In [ ]:
def validate_and_override_patterns(metadata):
    print("\n=== VALIDASI POLA LLM SEBELUM DIBERSIHKAN ===")
    print("Hanya tekan [ENTER] jika pola sudah benar. Jika salah, ketik pola yang benarnya.")
    
    print(f"\n[1] HEADER GENAP saat ini: {repr(metadata.get('header_even'))}")
    h_even_input = input("Update/Enter: ").strip()
    if h_even_input: 
        metadata['header_even'] = h_even_input
        
    print(f"\n[2] FOOTER GENAP saat ini: {repr(metadata.get('footer_even'))}")
    f_even_input = input("Update/Enter: ").strip()
    if f_even_input: 
        metadata['footer_even'] = f_even_input

    print(f"\n[3] HEADER GANJIL saat ini: {repr(metadata.get('header_odd'))}")
    h_odd_input = input("Update/Enter: ").strip()
    if h_odd_input: 
        metadata['header_odd'] = h_odd_input
        
    print(f"\n[4] FOOTER GANJIL saat ini: {repr(metadata.get('footer_odd'))}")
    f_odd_input = input("Update/Enter: ").strip()
    if f_odd_input: 
        metadata['footer_odd'] = f_odd_input
        
    return metadata

### 2. Teks Cleaning (Regex Lanjutan + AI Pattern)

In [ ]:
def clean_academic_text(pages_text, metadata):
    cleaned_full_text = ""
    
    patterns_to_remove = [
        metadata.get('header_even'), metadata.get('footer_even'), 
        metadata.get('header_odd'), metadata.get('footer_odd')
    ]
    valid_patterns = [p for p in patterns_to_remove if p and len(p.strip()) > 3]
    
    for p_text in pages_text:
        cleaned_page = p_text
        
        # 1. Hapus Pola Dinamis dari LLM (Mecah string berbasis baris agar lebih tahan banting)
        for pattern in valid_patterns:
            # Filter proteksi: abaikan pola dari LLM kalau dia menangkap caption "Figure" atau "Table" 
            # (karena caption figure beda-beda tiap halaman, nggak boleh dipakai re.sub global)
            if bool(re.search(r'^(Figure|Fig\.|Table|Tabel)\s*\d+', pattern.strip(), re.IGNORECASE)):
                continue 

            # Pecah menjadi part yang lebih kecil agar lisensi panjang tetap terhapus walaupun ada sedikit perbedaan whitespace
            parts = [part.strip() for part in pattern.split('\n') if len(part.strip()) > 5]
            for part in parts:
                escaped = re.escape(part)
                # Menghapus part tersebut di dalam teks
                cleaned_page = re.sub(f"^.*?{escaped}.*?$", "", cleaned_page, flags=re.MULTILINE | re.IGNORECASE)
                
        # 2. Hapus elemen gambar, grafik, dan tabel yang ter-extract ke teks (PENTING!)
        # Ini harus memakan newline juga kalau captionnya panjang
        cleaned_page = re.sub(r'^(Figure|Fig\.|Table|Tabel|Gambar)\s*\d+[:.].{0,200}$', '', cleaned_page, flags=re.MULTILINE | re.IGNORECASE)
        
        # 3. Hardcoded Regex Fixes untuk masalah footer spesifik yang persisten
        # Menghapus DOI
        cleaned_page = re.sub(r'DOI\s*:?\s*10\.\d{4,9}/[-._;()/:A-Za-z0-9]+', '', cleaned_page, flags=re.IGNORECASE)
        # Menghapus Lisensi Wiley & Creative Commons panjang
        cleaned_page = re.sub(r'Downloaded from https://onlinelibrary\.wiley\.com.*?(Creative Commons License|rules of use)', '', cleaned_page, flags=re.DOTALL | re.IGNORECASE)
        cleaned_page = re.sub(r'This work is licensed under a Creative Commons.*', '', cleaned_page, flags=re.IGNORECASE)
        # Menghapus info Corresponding Author (sering memecah chunk)
        cleaned_page = re.sub(r'^.*?Corresponding author:.*?$', '', cleaned_page, flags=re.MULTILINE | re.IGNORECASE)
        cleaned_page = re.sub(r'^.*?Email:.*?@.*$', '', cleaned_page, flags=re.MULTILINE | re.IGNORECASE)
        # Menghapus Majalah Obat Tradisional
        cleaned_page = re.sub(r'^.*Majalah Obat Tradisional.*$', '', cleaned_page, flags=re.MULTILINE | re.IGNORECASE)
        # Menghapus nomor halaman angka murni
        cleaned_page = re.sub(r'^\s*\d+\s*$', '', cleaned_page, flags=re.MULTILINE)
        
        # Normalisasi spasi putih
        cleaned_page = re.sub(r'\n\s*\n+', '\n\n', cleaned_page)
        cleaned_full_text += cleaned_page.strip() + "\n\n"
        
    # 4. Potong noise ekor (Daftar Pustaka, Konflik Kepentingan, dll.)
    abstract_match = re.search(r'\b(ABSTRACT|ABSTRAK|INTRODUCTION)\b', cleaned_full_text, re.IGNORECASE)
    if abstract_match:
        cleaned_full_text = cleaned_full_text[abstract_match.start():]

    # Pengetatan Regex referensi: harus menempel dengan awal newline, bukan sembarang kata di tengah kalimat!
    references_match = re.search(r'\n\s*(REFERENCES?|DAFTAR\s+PUSTAKA|BIBLIOGRAPHY|CONFLICT\s+OF\s+INTEREST|ACKNOWLEDGMENTS?)\s*\n', cleaned_full_text, re.IGNORECASE)
    if references_match:
        cleaned_full_text = cleaned_full_text[:references_match.start()].strip()
        
    return cleaned_full_text


### 3. Splitting/Chunking Engine

In [ ]:
def chunk_by_characters(text, chunk_size=1500, chunk_overlap=150):
    """
    Chunking per Karakter (Tergantung spasi dan newline, diukur berdasar panjang karakter absolut).
    Angka 1500 Karakter kasarnya setara dengan ~250-350 kata (Token), 
    jadi sangat aman dari memori limit Token.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    return splitter.split_text(text)

def chunk_by_words(text, chunk_size=500, chunk_overlap=50):
    """
    Chunking per Kata (Max 500 token).
    """
    splitter = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_text(text)

def chunk_by_sentences(text, sentences_per_chunk=15, overlap_sentences=3):
    """
    Chunking per-15 kalimat.
    """
    import re
    
    # Split berdasarkan titik/tanda seru/tanya yang diikuti spasi atau newline
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 10]
    
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk - overlap_sentences):
        chunk_sentences = sentences[i : i + sentences_per_chunk]
        chunk_text = ' '.join(chunk_sentences)
        
        if len(chunk_text.strip()) > 0:
            chunks.append(chunk_text)
            
        if i + sentences_per_chunk >= len(sentences):
            break
            
    return chunks


### 4. Vector Extraction Engine

In [ ]:
print("Memuat model embedding BAAI/bge-small-en-v1.5...")
# Beritahu library bila tak mau mengunduh ulang model setiap kali runtime direstart 
model_bge = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("Model Embedding Load OK!")

def extract_and_store_vectors(journal_uuid, chunks_list, table_name, count_column_name):
    if not journal_uuid or len(chunks_list) == 0:
        print(f"Skip {table_name}: Data tidak lengkap.")
        return
        
    start_time = time.time()
    print(f"-> Mengkonversi {len(chunks_list)} teks ke Vektor pada {table_name}...")
    vectors = model_bge.encode(chunks_list, normalize_embeddings=True).tolist()
    print(f"   Vektor selesai dalam {time.time() - start_time:.2f} detik.")

    data_to_insert = []
    for i, (text, vec) in enumerate(zip(chunks_list, vectors)):
        count_val = len(text.split()) if count_column_name == "token_count" else (len(text.split(".")) if count_column_name == "sentence_count" else len(text))
        
        row = {
            "journal_id": journal_uuid,
            "chunk_index": i + 1,
            "text_content": text,
            "embedding": vec,
            count_column_name: count_val
        }
        data_to_insert.append(row)
        
    try:
        response = supabase_client.table(table_name).insert(data_to_insert).execute()
        print(f"   ✅ Sukses menyimpan {len(data_to_insert)} baris!\n")
    except Exception as e:
        print(f"   ❌ Error insert ke {table_name}: {e}\n")


### FASE 1: MAIN PREPROCESSING (Eksperimen Pembersihan & Pemotongan)

In [ ]:
# TEST SCRIPT - PREPROCESSING:
# Ganti TEST_PDF_URL dengan URL File dari Storage Supabase Anda
TEST_PDF_URL = "https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf"

print("=== FASE 1: PREPROCESSING ===")
pages_raw = download_and_extract_text(TEST_PDF_URL)
metadata = extract_metadata_and_patterns(pages_raw)

print("\n[HASIL METADATA LLM]")
print(json.dumps(metadata, indent=2))

# JEDA UNTUK VALIDASI MANUAL!
metadata = validate_and_override_patterns(metadata)

clean_text = clean_academic_text(pages_raw, metadata)
print(f"\n[STATUS CLEANING] Sebelum: {sum(len(p) for p in pages_raw)} char -> Sesudah: {len(clean_text)} char")

print("\n--- Mengeksekusi Chunking ---")
character_chunks = chunk_by_characters(clean_text)
word_chunks = chunk_by_words(clean_text)
sentence_chunks = chunk_by_sentences(clean_text)

print(f"\nDihasilkan: ")
print(f"-> {len(character_chunks)} Character Chunks")
print(f"-> {len(word_chunks)} Word Chunks")
print(f"-> {len(sentence_chunks)} Sentence Chunks")


### FASE 2: DATABASE & VECTORIZATION (Sesi Penyimpanan)

In [ ]:
# TEST SCRIPT - DATABASE & EMBEDDING:
# Pastikan cell "FASE 1" di atas sudah di-running terlebih dahulu

print(f"=== FASE 2: DATABASE & VECTORIZATION ===")
file_name_target = TEST_PDF_URL.split("/")[-1]

if supabase_client is None:
    print("[PERINGATAN] Sesi Database dilewati. Harap konfigurasikan SUPABASE_URL jika ingin menyimpan hasil.")
else:
    try:
        # Cek Eksistensi Jurnal
        eksis_response = supabase_client.table("journals_experiment").select("id").eq("file_name", file_name_target).execute()
        journal_id = None

        if len(eksis_response.data) > 0:
            journal_id = eksis_response.data[0]['id']
            print(f"🔹 Jurnal sudah direkam. Melanjutkan operasi pada ID: {journal_id}")
        else:
            print("Ditemukan Jurnal Baru. Menyimpan detail jurnal...")
            journal_record = {
                "file_name": file_name_target,
                "title": metadata.get("title", "Unknown Title"),
                "authors": metadata.get("authors", "Unknown Authors"),
                "year": metadata.get("year", ""),
                "full_text": clean_text
            }
            response_jurnal = supabase_client.table("journals_experiment").insert(journal_record).execute()
            if len(response_jurnal.data) > 0:
                journal_id = response_jurnal.data[0]['id']
                print(f"✅ Jurnal BERHASIL disimpan. Menggunakan ID: {journal_id}")
                
        if journal_id:
            print("\n=== PROSES EXTRAKSI VEKTOR DIMULAI ===")
            extract_and_store_vectors(journal_id, character_chunks, "chunk_character", "char_count")
            extract_and_store_vectors(journal_id, word_chunks, "chunk_word", "token_count")
            extract_and_store_vectors(journal_id, sentence_chunks, "chunk_sentence", "sentence_count")
            print("🎉 SELURUH PROSES EXPERIMENT SELESAI!")
    except Exception as e:
        print(f"[ERROR] Operasi penyimpanan gagal. Error: {e}")